# Study report: systick-priority

The study measures how much of the IRQ tail under timer load comes from
the SysTick ISR. The driver gives SysTick the same priority as the
stimulus IRQ (EXTI0). The study asks if a lower SysTick priority makes
the IRQ tail, the IRQ max and the ISR to thread wake smaller. It has no
experiments of its own. It reads four experiments with imports.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-load-scaling@nucleo_g474re/st-systick-priority"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## The mechanism

The study uses these facts of the code (`study.md`):

1. The driver gives SysTick the same priority as the stimulus IRQ:
   SysTick and EXTI0 at priority 0. If both are pending, SysTick goes
   first, and EXTI0 cannot preempt it.
2. With `LATBENCH_SYSTICK_PRIO=1`, SysTick has priority 1. Then EXTI0
   can preempt the SysTick ISR outside its locked sections.
3. The locked sections of the SysTick ISR (timeout lock, clock lock)
   mask EXTI0 through BASEPRI at all priorities.
4. PendSV has a lower priority than SysTick at both priorities. Thus
   PendSV still waits for the SysTick ISR.

The SysTick ISR runs the callbacks of the load timers and restarts
them.

## H1 a: the IRQ tail over the load timers (wake-timers-1m)

The study `zephyr-versions` owns `wake-timers-1m`. Its points have 0, 16,
64 and 256 periodic load timers (10..100 ms), with time slicing
(`default`) and without (`noslice`). SysTick has the same priority as
EXTI0. H1 a expects that the `irq` p99.99 increases in the order of the
timers. The points `t256far` (256 one-shot timers one hour away) have
the same number as `t256` on the axis, so they are in the table only.

In [ ]:
if nbfig.built(inst, "wake-timers-1m"):
    tm = inst.rows("wake-timers-1m")
    irq = tm[(tm.metric == "irq") & ~tm.point.str.startswith("t256far")]
    nbfig.sweep(irq, "value.timers", "IRQ entry tail over the load timers (wake-timers-1m)",
                stats=("p99.99", "max"), group="axis.anchor").show()
    display(nbfig.pivot(tm[tm.metric == "irq"], ["p50", "p99", "p99.99", "max"]))

## H1 b and H2 a: the SysTick priority and the IRQ latency (wake-systick-prio-1m)

The points are the cube `timers` x `prio`: 64 or 256 periodic load
timers, and SysTick at the priority of EXTI0 (`same`, the driver
default) or at priority 1 (`lower`). These loads gave the IRQ tail in
`wake-timers-1m`.

- H1 b expects a smaller `irq` p99.99 with `lower` than with `same`: at
  priority 1, EXTI0 can preempt the SysTick ISR outside its locked
  sections.
- H2 a expects that the `irq` max stays about the same: the locked
  sections still mask EXTI0. The check is manual: the design gives no
  limit for "about the same".

The colour is the SysTick priority, the symbol is the statistic.

In [ ]:
if nbfig.built(inst, "wake-systick-prio-1m"):
    sp = inst.rows("wake-systick-prio-1m")
    nbfig.dots(sp[sp.metric == "irq"], "IRQ entry: SysTick priority same against lower", x="axis.timers",
               group="axis.prio", stats=("p50", "p99", "p99.99", "max")).show()
    t = sp[sp.metric == "irq"].pivot_table(index="axis.timers", columns="axis.prio",
                                            values=["p99", "p99.99", "max"], aggfunc="first", sort=False)
    for st in ("p99", "p99.99", "max"):
        t[(st, "lower / same")] = (t[(st, "lower")] / t[(st, "same")]).round(3)
    display(t[[(st, p) for st in ("p99", "p99.99", "max") for p in ("same", "lower", "lower / same")]])

In [ ]:
if nbfig.built(inst, "wake-systick-prio-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "wake-systick-prio-1m"), "wake-systick-prio-1m: CCDF",
               ["irq", "isr2thr", "irq2thr"]).show()

## H2 b: the SysTick priority and the ISR to thread wake

H2 b expects that `isr2thr` does not change much: PendSV has a lower
priority than SysTick at both priorities, so PendSV still waits for the
SysTick ISR. The check is manual: the design gives no limit for "not
much".

In [ ]:
if nbfig.built(inst, "wake-systick-prio-1m"):
    nbfig.dots(sp[sp.metric == "isr2thr"], "ISR to thread: SysTick priority same against lower", x="axis.timers",
               group="axis.prio", stats=("p50", "p99", "p99.99", "max")).show()
    t = sp[sp.metric == "isr2thr"].pivot_table(index="axis.timers", columns="axis.prio",
                                                values=["p50", "p99", "p99.99", "max", "mean"], aggfunc="first",
                                                sort=False)
    for st in ("p50", "p99", "p99.99", "max", "mean"):
        t[(st, "lower - same")] = t[(st, "lower")] - t[(st, "same")]
    display(t[[(st, p) for st in ("p50", "p99", "p99.99", "max", "mean") for p in ("same", "lower", "lower - same")]])

## The spread of the t64 tail over seeds (wake-stim-seed-1m, wake-load-seed-tail)

The study `measurement-validity/stimulus-sampling` owns both
experiments. They measure the `t64` point over 8 stimulus seeds and 4
load seeds (one of them a run of 4M samples). They have hot code in the
CCM SRAM (`LATBENCH_CODE_RAM`), and the points of `wake-systick-prio-1m`
do not. The figure shows how much the `irq` tail changes with the
seeds alone. Compare this spread with the difference between `same` and
`lower` above.

In [ ]:
parts = [inst.rows(e).assign(experiment=e) for e in ("wake-stim-seed-1m", "wake-load-seed-tail")
         if nbfig.built(inst, e)]
if parts:
    seeds = pd.concat(parts)
    seeds = seeds[seeds.metric == "irq"]
    nbfig.dots(seeds, "IRQ entry tail at t64 over the seeds", stats=("p99", "p99.99", "max"),
               group="experiment").show()
    display(seeds.set_index(["experiment", "point"])[["n", "p50", "p99", "p99.9", "p99.99", "max"]])
    display(seeds.groupby("experiment")[["p99", "p99.99", "max"]].agg(["min", "max"]))